<a href="https://colab.research.google.com/github/sneakersaddict/Automated-web-app-vuln-log-audit-assistant/blob/main/notebooks/ines_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/sneakersaddict/Automated-web-app-vuln-log-audit-assistant.git
%cd Automated-web-app-vuln-log-audit-assistant
!git config user.email "bouzekri.ines@gmail.com"
!git config user.name "Ines"
!pip install -q datasets

Mounted at /content/drive
Cloning into 'Automated-web-app-vuln-log-audit-assistant'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 41 (delta 10), reused 35 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 224.10 KiB | 9.34 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Automated-web-app-vuln-log-audit-assistant


In [ ]:
!ls data src

data:
logs_150.csv  raw  test_20.csv	train.csv  val.csv

src:
build_dataset.py	 evaluate.py  train_planb.py
classifier_planb.joblib  features.py  triage.py
classifier.py		 __init__.py  tune_triage.py


In [2]:
import pandas as pd, torch, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from src.features import request_text

train = pd.read_csv("data/train.csv"); val = pd.read_csv("data/val.csv")
name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
lab = {"benign": 0, "malicious": 1}

def prep(df):
    d = Dataset.from_dict({"text": df.log_line.map(request_text).tolist(), "label": df.label.map(lab).tolist()})
    return d.map(lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=128), batched=True)

tr, va = prep(train), prep(val)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)

def metrics(p):
    return {"acc": float((p.predictions.argmax(1) == p.label_ids).mean())}

args = TrainingArguments("models/ft", num_train_epochs=10, per_device_train_batch_size=16,
                         learning_rate=5e-5, eval_strategy="epoch", save_strategy="no", report_to="none")
# on older transformers versions, replace eval_strategy with evaluation_strategy
trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va, compute_metrics=metrics)
trainer.train()
print("TRAIN:", trainer.evaluate(tr))
print("VAL:  ", trainer.evaluate(va))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

... (5 outputs left out)

Training Loss,Validation Loss,Epoch,Acc
No log,0.006730,10,1.000000


TRAIN: {'eval_loss': 0.006730475928634405, 'eval_acc': 1.0}


Training Loss,Validation Loss,Epoch,Acc
No log,0.009571,10,1.000000


VAL:   {'eval_loss': 0.009571314789354801, 'eval_acc': 1.0}


In [3]:
import json, os, torch
logits = torch.tensor(trainer.predict(va).predictions); y = torch.tensor(va["label"])
logT = torch.nn.Parameter(torch.zeros(1))
opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
def closure():
    opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(logits / logT.exp(), y)
    loss.backward()
    return loss
opt.step(closure)
T = float(logT.exp().clamp(0.25, 5.0))
print("T =", T)

probs = torch.softmax(logits / T, dim=1)
print("val acc:", (probs.argmax(1) == y).float().mean().item(),
      "| avg confidence:", probs.max(1).values.mean().item())

out = "/content/drive/MyDrive/soc_model"
json.dump({"T": T}, open(out + "/temperature.json", "w"))

/tmp/ipykernel_701/2893483709.py:11: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  T = float(logT.exp().clamp(0.25, 5.0))


T = 0.25
val acc: 1.0 | avg confidence: 0.9999931454658508


In [4]:
T = 1.0
json.dump({"T": T}, open(out + "/temperature.json", "w"))

In [5]:
!sed -i 's/for b in \[0.80, 0.90, 0.95\]/for b in [0.60, 0.70, 0.80, 0.90]/' src/tune_triage.py

In [7]:
import json
out = "/content/drive/MyDrive/soc_model"
trainer.save_model(out); tok.save_pretrained(out)
json.dump({"T": 1.0}, open(out + "/temperature.json", "w"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
os.environ["FT_MODEL_DIR"] = out
!python -m src.tune_triage

Loading weights: 100% 104/104 [00:00<00:00, 2513.57it/s]
val accuracy: 1.0
benign_thr mal_thr | malicious_auto_closed benign_auto_closed alerts review
0.60 0.70 | 0 12 11 7
0.60 0.85 | 0 12 11 7
0.60 0.95 | 0 12 11 7
0.70 0.70 | 0 12 11 7
0.70 0.85 | 0 12 11 7
0.70 0.95 | 0 12 11 7
0.80 0.70 | 0 12 11 7
0.80 0.85 | 0 12 11 7
0.80 0.95 | 0 12 11 7
0.90 0.70 | 0 11 11 8
0.90 0.85 | 0 11 11 8
0.90 0.95 | 0 11 11 8


In [9]:
!sed -i 's/^BENIGN_THR = .*/BENIGN_THR = 0.80   # tuned on val.csv/' src/triage.py
!grep THR src/triage.py

BENIGN_THR = 0.80   # tuned on val.csv
MAL_THR = 0.85
def triage(label, confidence, attack_type=None, benign_thr=BENIGN_THR, mal_thr=MAL_THR):


In [10]:
!git add src/
!git commit -m "Tune triage thresholds on validation set"
!git pull --no-rebase --no-edit origin main
!git push

[main a11dafd] Tune triage thresholds on validation set
 2 files changed, 2 insertions(+), 2 deletions(-)
From https://github.com/sneakersaddict/Automated-web-app-vuln-log-audit-assistant
 * branch            main       -> FETCH_HEAD
Already up to date.
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 482 bytes | 482.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/sneakersaddict/Automated-web-app-vuln-log-audit-assistant.git
   5e5165e..a11dafd  main -> main


In [11]:
os.environ["FT_MODEL_DIR"] = out
!python -m src.evaluate

Loading weights: 100% 104/104 [00:00<00:00, 2868.03it/s]
test accuracy: 0.95 (19/20, target >= 16/20)
[[13  1]
 [ 0  6]]
 id     label   attack_type      pred  conf            decision
 48 malicious          SQLi malicious 0.996 HIGH_PRIORITY_ALERT
 33 malicious          SQLi malicious 0.996 HIGH_PRIORITY_ALERT
 75 malicious          SQLi malicious 0.996 HIGH_PRIORITY_ALERT
 49 malicious          SQLi    benign 0.985          AUTO_CLOSE
119 malicious          SQLi malicious 0.996 HIGH_PRIORITY_ALERT
 73 malicious           XSS malicious 0.996     FLAG_FOR_REVIEW
 67 malicious           XSS malicious 0.996     FLAG_FOR_REVIEW
 91 malicious           XSS malicious 0.996     FLAG_FOR_REVIEW
 70 malicious           XSS malicious 0.996     FLAG_FOR_REVIEW
138 malicious           XSS malicious 0.996     FLAG_FOR_REVIEW
  8 malicious PathTraversal malicious 0.995 HIGH_PRIORITY_ALERT
 50 malicious PathTraversal malicious 0.996 HIGH_PRIORITY_ALERT
 81 malicious PathTraversal malicious 0.984 HIG

In [12]:
!git add results/
!git commit -m "Add final classifier results"
!git pull --no-rebase --no-edit origin main
!git push

[main c000819] Add final classifier results
 1 file changed, 21 insertions(+)
 create mode 100644 results/classifier_results.csv
From https://github.com/sneakersaddict/Automated-web-app-vuln-log-audit-assistant
 * branch            main       -> FETCH_HEAD
Already up to date.
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.29 KiB | 2.29 MiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/sneakersaddict/Automated-web-app-vuln-log-audit-assistant.git
   a11dafd..c000819  main -> main


In [13]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
r = pd.read_csv("results/classifier_results.csv")
print(r[r.id == 49][["log_line", "attack_type", "pred", "conf"]].to_string())
print(r[r.source == "manual"][["id", "label", "attack_type", "pred", "decision", "log_line"]].to_string())

                                                                                                                                                                                                                                                                                                                                                                                                                     log_line attack_type    pred   conf
3  192.168.152.193 - - [13/Nov/2026:13:34:47 +0000] "POST /tienda1/miembros/editar.jsp?modo=registro&login=lemar&password=43lpe7de2o&nombre=Bianca&apellidos=Valor+Agudo&email=vallarine7%407horas.mo&dni=90125861Q&direccion=Plaza+Monasterio+De+La+Oliva%2C+S%2FN%2C+&ciudad=Tembleque&cp=20493&provincia=Navarra&ntc=7119531551958366&B1=Registrar%3C%21--%23include+file%3D%22archivo_secreto%22+--%3E HTTP/1.1" 403 4673        SQLi  benign  0.985
     id      label    attack_type       pred             decision                                                     

In [14]:
!git add results/
!git commit -m "Add final classifier results"
!git pull --no-rebase --no-edit origin main
!git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
From https://github.com/sneakersaddict/Automated-web-app-vuln-log-audit-assistant
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date


In [15]:
from src.features import request_text
line = r[r.id == 49].log_line.iloc[0]
t = request_text(line)
print(len(t), "chars;", len(tok(t)["input_ids"]), "tokens (limit 128)")
print(t)

350 chars; 182 tokens (limit 128)
POST /tienda1/miembros/editar.jsp?modo=registro&login=lemar&password=43lpe7de2o&nombre=Bianca&apellidos=Valor+Agudo&email=vallarine7%407horas.mo&dni=90125861Q&direccion=Plaza+Monasterio+De+La+Oliva%2C+S%2FN%2C+&ciudad=Tembleque&cp=20493&provincia=Navarra&ntc=7119531551958366&B1=Registrar%3C%21--%23include+file%3D%22archivo_secreto%22+--%3E HTTP/1.1
